In [1]:
import os
from dotenv import load_dotenv

#加载.env文件
load_dotenv()

#定义搜索工具
from langchain_tavily import TavilySearch

web_search = TavilySearch(
    api_key=os.getenv('Tavily_api_key'),
    max_results = 5,
    topic = "general",
)


添加记忆管理

In [3]:
#添加记忆管理
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

#连接sqlite
connection = sqlite3.connect('resources/personal_chief.db',check_same_thread=False)
#初始化checkpoint
checkpointer = SqliteSaver(connection)
#自动建表
checkpointer.setup()


In [4]:
from langchain.chat_models import init_chat_model
#构建model
model = init_chat_model(
    model="qwen3.8-flash",
    model_provider="openai",
    api_key=os.getenv("api_key"),
    base_url=os.getenv("base_url")
)


In [5]:
from langchain.agents import create_agent

#定义提示词
system_prompt ='''
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行食谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先使用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。

'''

#定义agent
agent = create_agent(
    model = model,
    tools = [web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

In [6]:
from langchain_core.messages import HumanMessage

multimodal_message = HumanMessage([
    {"type":"text","text":"帮我看看能做什么菜？"},
    {"type":"image","url":"https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg"}
])
config ={"configurable":{"thread_id":"1"}}

In [7]:
response = agent.invoke(
    {"messages":[multimodal_message]},
    config
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么菜？'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]
================================== Ai Message ==================================

我先仔细辨认照片里的食材，然后去检索匹配的家常食谱。

**第一步：识别与评估食材**

从冰箱照片（拍摄于 2022-04-03）中，我辨认出以下食材，并评估了新鲜度与可用量：

| 类别 | 食材 | 状态/可用量评估 |
|---|---|---|
| 蛋白类 | 嫩豆腐（碗中泡水）1块、虾仁（玻璃碗装，已解冻）约200g | 豆腐需尽快用、虾仁已解冻建议当天食用 |
| 蔬菜·深色 | 西兰花 1颗、茄子 1根（紫亮）| 新鲜，可用 |
| 蔬菜·椒类 | 番茄 3个、尖青椒 2根、圆青椒 1个 | 番茄熟度正好，青椒饱满 |
| 蔬菜·豆类 | 四季豆/长豆角 1把 | 新鲜脆嫩 |
| 蔬菜·叶菜 | 大白菜/包菜 1棵、生菜/油麦菜 数棵、香菜 1小把 | 叶菜偏多，需优先消耗 |
| 蔬菜·其他 | 黄瓜 1根、金针菇 1袋、蒜薹/小葱 1把、白洋葱 2个、盒装小番茄/葡萄 1盒 | 金针菇袋装未开封，可放 |
| 调味/腌菜 | 腊肠笋子酱菜 1瓶、番茄酱/辣椒酱 1瓶 | 佐餐或提味用 |

现在我去检索能组合这些食材的家常食谱。
Tool Calls:
  tavily_search (call_0c3162aaf5824171b67a4f0b)
 Call ID: call_0c3162aaf5824171b67a4f0b
  Args:
    query: 虾仁 豆腐 西兰花 家常菜谱 做法
    search_depth: advanced
  tavily_search (call_cee1b37681894123b42